# TransferGuard — Negative Transfer Experiment 1

**CIFAR-10 | 80% source-label corruption | Local GPU (Windows / RTX 2050)**

This notebook tests a *specific, research-backed failure condition*: a CNN trained on unreliable source labels can learn features that are harmful to reuse on a clean target task.

**Based on:** Xu et al., *Residual Feature Integration is Sufficient to Prevent Negative Transfer*, ICLR 2026. [Paper](https://proceedings.iclr.cc/paper_files/paper/2026/hash/7d998ff3e6ecce821986c9dd876be9bc-Abstract-Conference.html) · [Author code (`noise.py`)](https://github.com/Xyc-arch/ReFine-residual-feature-integration/blob/main/noise.py) · [Model definitions](https://github.com/Xyc-arch/ReFine-residual-feature-integration/blob/main/model_def_test10/model_def.py)

### What we will train

1. **Source (BigCNN):** 10,000 CIFAR-10 images; 80% of source labels replaced by a *different wrong label on each access*; 60 epochs.
2. **Scratch (SmallCNN):** a new, smaller CNN on 4,000 **clean, non-overlapping** CIFAR-10 target images; 30 epochs.
3. **Frozen transfer (BigCNN):** reuse the corrupted-source CNN; freeze everything except the existing final linear layer, as in the released reference; train 30 epochs on the same 4,000 clean target images.
4. **Optional fairness checks:** scratch BigCNN of the same architecture, and full fine-tuning of the corrupted BigCNN.

The source model and scratch baseline use **different CNN sizes in the original paper**. The first comparison is for exploring/reproducing the published failure mechanism, **not a clean architecture-matched proof that full fine-tuning fails**. To make the broader claim, run the optional checks and repeat seeds.

**Important:** A negative-transfer result is *not guaranteed*. We only inspect the official CIFAR-10 test set in the final evaluation cell. Validation and 500 reserved calibration examples come exclusively from the 50,000-image training split.

## Step 1 — Imports and experiment settings

Install **CUDA-enabled PyTorch and TorchVision** into your local Python environment before running. In a Jupyter cell you may install missing analysis libraries with `!pip install numpy pandas matplotlib`, but do **not** replace a working CUDA PyTorch installation with CPU wheels.

In VS Code: select the interpreter/kernel that already reports `torch.cuda.is_available() == True`. If you are using a Jupyter kernel, install `ipykernel` into that environment.

In [ ]:
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms

seed = 42
source_epochs = 60       # Paper-inspired full schedule
Target_epochs = 30
batch_size = 16          # Local RTX 2050 4 GB: increase to 32 if it fits
noise_rate = 0.80
learning_rate = 0.01

# Optional, slower scientific controls (run after the main comparison).
RUN_MATCHED_SCRATCH = False
RUN_FULL_FINETUNE = False

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'notebooks').is_dir():
    raise RuntimeError('Start Jupyter from the repository root or notebooks directory.')
folder = root / 'transferguard_cifar10_results'
data_dir = root / 'data'
folder.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

def set_seed(value):
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(value)

set_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('CUDA is not available. Select your CUDA-enabled local Python kernel.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Results folder:', folder.resolve())

## Step 2 — Download CIFAR-10 and create disjoint splits

We follow the authors' 10,000 source / 4,000 target sizes, with **2,000 validation and 500 calibration images additionally held out** from the official training partition. The calibration images are **not used in this notebook**; they are for developing TransferGuard later. The official 10,000-image test partition is evaluated only at the end.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_data = datasets.CIFAR10(root=str(data_dir), train=True,
                              download=True, transform=transform)
test_data = datasets.CIFAR10(root=str(data_dir), train=False,
                             download=True, transform=transform)

rng = np.random.RandomState(seed)
indices = rng.permutation(len(train_data))
source_ids = indices[:10000]
target_ids = indices[10000:14000]
val_ids = indices[14000:16000]
cal_ids = indices[16000:16500]

assert len(set(source_ids) | set(target_ids) | set(val_ids) | set(cal_ids)) == 16500
np.savez(folder / 'split_indices_seed42.npz', source=source_ids,
         target=target_ids, validation=val_ids, calibration=cal_ids)

source_clean = Subset(train_data, source_ids)
target_data = Subset(train_data, target_ids)
val_data = Subset(train_data, val_ids)
cal_data = Subset(train_data, cal_ids)

print('Source:', len(source_clean), '| Target:', len(target_data),
      '| Validation:', len(val_data), '| Guard calibration:', len(cal_data),
      '| Official test:', len(test_data))

## Step 3 — Define the two CNNs from the paper

Both architectures below use the layer dimensions specified in the authors' [`model_def_test10/model_def.py`](https://github.com/Xyc-arch/ReFine-residual-feature-integration/blob/main/model_def_test10/model_def.py). The source BigCNN is larger, and the target-only baseline SmallCNN is smaller. This is why a **same-architecture scratch BigCNN** is available later as an optional comparison.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        return self.fc_layers(x.flatten(1))


class BigCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 80, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(80, 160, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(160, 320, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(320, 640, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(640, 640, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(640, 768, 3, padding=1), nn.ReLU()
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(768, 2560), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(2560, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        return self.fc_layers(x.flatten(1))


print('SmallCNN parameters:', sum(p.numel() for p in SmallCNN().parameters()))
print('BigCNN parameters:', sum(p.numel() for p in BigCNN().parameters()))
with torch.no_grad():
    assert SmallCNN()(torch.randn(2, 3, 32, 32)).shape == (2, 10)
    assert BigCNN()(torch.randn(2, 3, 32, 32)).shape == (2, 10)
print('Forward-pass checks passed')

## Step 4 — Dynamically corrupt the source labels

**Important replication detail:** for each image access, with probability 0.80 we return a random **wrong** class label. The wrong label is resampled every time; target, validation, and test labels stay clean. This matches the mechanism in the reference `noise.py` rather than assigning one permanently wrong label to each source image.

In [ ]:
class NoisyLabels(Dataset):
    def __init__(self, clean_dataset, rate=0.8):
        self.data = clean_dataset
        self.rate = rate

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        image, label = self.data[index]
        if np.random.rand() < self.rate:
            wrong = np.random.randint(0, 9)
            if wrong >= label:
                wrong += 1
            label = int(wrong)
        return image, label


source_data = NoisyLabels(source_clean, noise_rate)

def make_loader(data, shuffle=False):
    return DataLoader(data, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=True)

source_loader = make_loader(source_data, shuffle=True)
target_loader = make_loader(target_data, shuffle=True)
val_loader = make_loader(val_data)
# test_loader is deliberately created only in the last cell.

sample_x, sample_y = source_data[0]
assert sample_x.shape == (3, 32, 32) and 0 <= sample_y < 10
print('Source noise rate:', noise_rate)
print('A source label can change each time an image is fetched.')

## Step 5 — Simple training and validation functions

- Same SGD optimizer and learning rate as the released paper: **SGD(lr=0.01, momentum=0.9)**.
- Saves a checkpoint **every epoch**, so you can continue after an interruption.
- Also saves the best validation checkpoint; source pretraining later uses the **last** source checkpoint to match the research procedure.
- Training curves use validation data, **not** the official test set.

The added validation and checkpoint selection are *our extensions* to the reference implementation. The original paper evaluates the last target epoch. We will report which checkpoint choice was used.

In [ ]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        preds = model(images).argmax(1)
        correct += (preds == labels).sum().item()
        total += len(labels)
    return 100 * correct / total


def train_model(model, loader, name, epochs):
    model = model.to(device)
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=learning_rate, momentum=0.9
    )
    loss_fn = nn.CrossEntropyLoss()
    path = folder / f'{name}_last.pt'
    best_path = folder / f'{name}_best.pt'
    history = []
    start_epoch = 0
    best_acc = -1

    if path.exists():
        saved = torch.load(path, map_location='cpu', weights_only=True)
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        start_epoch = saved['epoch']
        history = saved['history']
        best_acc = saved['best_acc']
        print(f'Resuming {name} from epoch {start_epoch}')

    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0.0
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)

        train_loss = total_loss / len(loader.dataset)
        val_acc = accuracy(model, val_loader)
        history.append([epoch + 1, train_loss, val_acc])

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_path)

        torch.save({
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'epoch': epoch + 1,
            'history': history,
            'best_acc': best_acc
        }, path)
        pd.DataFrame(history, columns=['epoch', 'train_loss', 'val_acc']).to_csv(
            folder / f'{name}_history.csv', index=False
        )
        print(f'{name}: {epoch+1:02}/{epochs} | loss {train_loss:.3f} | val {val_acc:.2f}%')

    return model, pd.DataFrame(history, columns=['epoch', 'train_loss', 'val_acc'])


def trained_model(model_class, name, checkpoint='best'):
    model = model_class().to(device)
    if checkpoint == 'best':
        state = torch.load(folder / f'{name}_best.pt', map_location='cpu', weights_only=True)
    else:
        state = torch.load(folder / f'{name}_last.pt', map_location='cpu', weights_only=True)['model']
    model.load_state_dict(state)
    return model

## Step 6 — Pretrain the corrupted-source BigCNN (60 epochs)

This is the main source-training expense. The file `source_noise80_last.pt` saves progress each epoch; re-running this cell resumes it. The reference uses **the final source checkpoint**, not the best validation epoch. If your GPU runs out of memory, change `batch_size = 16` to `8` in Step 1 and re-run Step 4 and this cell. That changes the paper's batch size and should be reported.

In [ ]:
set_seed(seed)
source_model, source_history = train_model(
    BigCNN(), source_loader, 'source_noise80', source_epochs
)
print('Source checkpoint:', folder / 'source_noise80_last.pt')

del source_model
torch.cuda.empty_cache()

## Step 7 — Train SmallCNN from scratch (30 epochs)

This is the paper-style target-only baseline. It sees only the **4,000 clean target images**.

In [ ]:
set_seed(seed)
scratch_model, scratch_history = train_model(
    SmallCNN(), target_loader, 'scratch_small', Target_epochs
)
del scratch_model
torch.cuda.empty_cache()

## Step 8 — Frozen transfer from the corrupted source (30 epochs)

Load the **final epoch** of the corrupted-source BigCNN. Freeze all parameters *except* the existing last linear layer, following the authors' `train_linear_prob` code. This is a **last-layer-only adaptation**, not full fine-tuning. We leave the source-trained output head in place to match the original experimental path; its initialization is a confound to investigate later. The BigCNN has no batch-normalization layers; dropout remains active during head training, as in the reference code.

In [ ]:
set_seed(seed)
probe = trained_model(BigCNN, 'source_noise80', checkpoint='last')

for param in probe.parameters():
    param.requires_grad = False
for param in probe.fc_layers[-1].parameters():
    param.requires_grad = True

probe, probe_history = train_model(
    probe, target_loader, 'frozen_noise80', Target_epochs
)
del probe
torch.cuda.empty_cache()

## Step 9 — Compare the two core models using validation accuracy

**This is exploratory only**. The reference comparison uses a small scratch CNN versus a frozen BigCNN, so this gap alone is not an architecture-matched causal estimate of transfer benefit or harm. The official test data are not used yet.

In [ ]:
scratch_val = scratch_history.val_acc.max()
probe_val = probe_history.val_acc.max()
print(f'Scratch SmallCNN best validation: {scratch_val:.2f}%')
print(f'Frozen BigCNN best validation: {probe_val:.2f}%')
print(f'Validation gap (scratch - frozen): {scratch_val - probe_val:+.2f} points')

plt.figure(figsize=(7, 4))
plt.plot(scratch_history.epoch, scratch_history.val_acc, label='Scratch SmallCNN')
plt.plot(probe_history.epoch, probe_history.val_acc, label='Frozen corrupted BigCNN')
plt.xlabel('Epoch')
plt.ylabel('Clean validation accuracy (%)')
plt.title('CIFAR-10: source-label-corruption pilot')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(folder / 'validation_curves.png', dpi=150)
plt.show()

## Step 10 — Optional: same-architecture scratch BigCNN

**Run this if you want to claim an architecture-matched negative transfer result.** Without it, the published-style scratch-versus-frozen comparison uses different model sizes. It adds one 30-epoch target run.

In [ ]:
if RUN_MATCHED_SCRATCH:
    set_seed(seed)
    scratch_big, big_history = train_model(
        BigCNN(), target_loader, 'scratch_big', Target_epochs
    )
    del scratch_big
    torch.cuda.empty_cache()
    print('Same-architecture scratch BigCNN finished.')
else:
    print('Skipped. Set RUN_MATCHED_SCRATCH = True in Step 1 to enable.')

## Step 11 — Optional: full fine-tuning of corrupted BigCNN

This separates **harmful frozen features** from **harmful pretrained initialization**. Starting from the same corrupted-source checkpoint, train **all layers** for 30 epochs on the clean target task. If it beats scratch, you cannot say full fine-tuning is harmful just because the frozen probe failed.

In [ ]:
if RUN_FULL_FINETUNE:
    set_seed(seed)
    finetune = trained_model(BigCNN, 'source_noise80', checkpoint='last')
    for param in finetune.parameters():
        param.requires_grad = True
    finetune, ft_history = train_model(
        finetune, target_loader, 'finetune_noise80', Target_epochs
    )
    del finetune
    torch.cuda.empty_cache()
    print('Full fine-tuning completed.')
else:
    print('Skipped. Set RUN_FULL_FINETUNE = True in Step 1 to enable.')

## Step 12 — Final official test evaluation (run once)

Only run this after selecting all model conditions and checkpoint rules from validation. This cell evaluates the saved **best-validation checkpoints** (an extension of the paper's final-epoch protocol). The paper also evaluated multiple random seeds; this notebook is initially **one pilot seed**, and repeating it is needed for reliable conclusions.

In [ ]:
test_loader = make_loader(test_data)
model_specs = [
    ('scratch_small', SmallCNN),
    ('frozen_noise80', BigCNN)
]
if (folder / 'scratch_big_best.pt').exists():
    model_specs.append(('scratch_big', BigCNN))
if (folder / 'finetune_noise80_best.pt').exists():
    model_specs.append(('finetune_noise80', BigCNN))

rows = []
for name, model_class in model_specs:
    model = trained_model(model_class, name, checkpoint='best')
    score = accuracy(model, test_loader)
    rows.append({'method': name, 'test_accuracy': score})
    del model
    torch.cuda.empty_cache()

results = pd.DataFrame(rows)
results.to_csv(folder / 'test_results.csv', index=False)
print(results.to_string(index=False))

def test_acc(name):
    return float(results.loc[results.method == name, 'test_accuracy'].iloc[0])

paper_gap = test_acc('scratch_small') - test_acc('frozen_noise80')
print(f'\nPaper-style gap (Small scratch - Frozen BigCNN): {paper_gap:+.2f} points')
if 'scratch_big' in results.method.values:
    matched_gap = test_acc('scratch_big') - test_acc('frozen_noise80')
    print(f'Same-architecture gap (Big scratch - Frozen BigCNN): {matched_gap:+.2f} points')
if 'finetune_noise80' in results.method.values and 'scratch_big' in results.method.values:
    ft_gap = test_acc('scratch_big') - test_acc('finetune_noise80')
    print(f'Full-fine-tuning gap (Big scratch - Big fine-tune): {ft_gap:+.2f} points')

print('\nPositive gap means the transfer method scored below that scratch reference.')
print('A single pilot cannot establish repeatable negative transfer.')

plt.figure(figsize=(8, 4))
plt.bar(results.method, results.test_accuracy)
plt.ylabel('Test accuracy (%)')
plt.title('CIFAR-10 source corruption (80%)')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(folder / 'final_comparison.png', dpi=150)
plt.show()

## Next step — Build an honest TransferGuard benchmark

**Do not choose a detection rule by reading the final test accuracy above.** That would leak evaluation information into the research method. Instead:

1. Repeat the *same* design with a **0% clean-source control**, and ideally **40% source noise**, so that the guard must distinguish useful from harmful sources.
2. Perform 3 paired target seeds; keep architecture and clean-target splits matched. For a true source-variability estimate, repeat source pretraining too.
3. Investigate whether the harmful result persists with (a) a **freshly reset final head** and (b) an **architecture-matched scratch model**.
4. Reserve the 500-example **calibration set** for the proposed two-signal detection rule; evaluate that rule against an early-validation-only decision.
5. Only then make claims about **detecting** and **avoiding** negative transfer. This notebook measures the phenomenon; it does not implement the detector.

**What is / isn't a reproduction?** The paper's *layers, corruption mechanism, source/target sample counts, SGD settings and epochs* are retained. We changed **batch size for a 4 GB GPU**, added **held-out validation/calibration**, added **checkpoint saving**, and use **one run rather than five**. So this is a **research-inspired pilot**, not an exact numerical reproduction and no published accuracy is guaranteed.